In [2]:
# ============================================================
# WEATHER FORECASTING MODEL
# Used ONLY for drought reasoning
# ============================================================

import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ============================================================
# 1. LOAD HISTORICAL DATA
# ============================================================

DATA_PATH = r"C:\Users\niran\Downloads\Drought prediction_ML\data\train_timeseries.csv"

print("Loading dataset...")

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
print("Columns:")
print(df.columns.tolist())


# ============================================================
# 2. SELECT REQUIRED COLUMNS
# ============================================================

required_columns = [
    "fips",
    "date",
    "PRECTOT",
    "T2M",
    "QV2M"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

df = df[required_columns].copy()


# ============================================================
# 3. CLEAN DATA
# ============================================================

df["fips"] = (
    df["fips"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

weather_columns = [
    "PRECTOT",
    "T2M",
    "QV2M"
]

for col in weather_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    # NASA missing-value handling
    df.loc[
        df[col] <= -900,
        col
    ] = np.nan


df = df.dropna(
    subset=weather_columns
)

df = df.sort_values(
    ["fips", "date"]
).reset_index(drop=True)


print("\nAfter cleaning:", df.shape)


# ============================================================
# 4. START WITH ONE COUNTY
# ============================================================

TARGET_FIPS = "48001"

county_df = df[
    df["fips"] == TARGET_FIPS
].copy()

county_df = county_df.sort_values(
    "date"
).reset_index(drop=True)


print("\nCounty:", TARGET_FIPS)
print("County shape:", county_df.shape)

print(
    county_df.head()
)

print(
    county_df.tail()
)

# ============================================================
# 5. SEASONAL FEATURES
# ============================================================

county_df["month"] = (
    county_df["date"].dt.month
)

county_df["day_of_year"] = (
    county_df["date"].dt.dayofyear
)

# Cyclical representation of season

county_df["month_sin"] = np.sin(
    2 * np.pi *
    county_df["month"] / 12
)

county_df["month_cos"] = np.cos(
    2 * np.pi *
    county_df["month"] / 12
)

county_df["day_sin"] = np.sin(
    2 * np.pi *
    county_df["day_of_year"] / 365
)

county_df["day_cos"] = np.cos(
    2 * np.pi *
    county_df["day_of_year"] / 365
)


# ============================================================
# 6. LAG FEATURES
# ============================================================

LAGS = 30

for col in weather_columns:

    for lag in range(1, LAGS + 1):

        county_df[
            f"{col}_lag{lag}"
        ] = county_df[col].shift(lag)


# ============================================================
# 7. ROLLING WEATHER FEATURES
# ============================================================

for col in weather_columns:

    county_df[
        f"{col}_mean_7d"
    ] = (
        county_df[col]
        .rolling(7)
        .mean()
    )

    county_df[
        f"{col}_mean_14d"
    ] = (
        county_df[col]
        .rolling(14)
        .mean()
    )

    county_df[
        f"{col}_mean_30d"
    ] = (
        county_df[col]
        .rolling(30)
        .mean()
    )


# ============================================================
# 8. TARGET = NEXT DAY
# ============================================================

for col in weather_columns:

    county_df[
        f"target_{col}"
    ] = county_df[col].shift(-1)


# ============================================================
# 9. REMOVE MISSING ROWS
# ============================================================

county_df = (
    county_df
    .dropna()
    .reset_index(drop=True)
)

print(
    "\nData available for training:",
    county_df.shape
)


# ============================================================
# 10. CREATE FEATURE LIST
# ============================================================

features = []

# Lag features

for col in weather_columns:

    for lag in range(1, LAGS + 1):

        features.append(
            f"{col}_lag{lag}"
        )


# Rolling features

for col in weather_columns:

    features.extend([
        f"{col}_mean_7d",
        f"{col}_mean_14d",
        f"{col}_mean_30d"
    ])


# Seasonal features

features.extend([
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos"
])


X = county_df[features]


# ============================================================
# 11. CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================

split_index = int(
    len(county_df) * 0.80
)

X_train = X.iloc[
    :split_index
]

X_test = X.iloc[
    split_index:
]

print(
    "\nTraining rows:",
    len(X_train)
)

print(
    "Testing rows :",
    len(X_test)
)


# ============================================================
# 12. TRAIN MODELS
# ============================================================

weather_models = {}

weather_results = {}


for target in weather_columns:

    print("\n")
    print("=" * 70)
    print(
        f"TRAINING WEATHER MODEL — {target}"
    )
    print("=" * 70)

    target_column = (
        f"target_{target}"
    )

    y = county_df[target_column]

    y_train = y.iloc[
        :split_index
    ]

    y_test = y.iloc[
        split_index:
    ]

    start_time = time.time()

    model = RandomForestRegressor(

        n_estimators=150,

        max_depth=15,

        min_samples_split=2,

        min_samples_leaf=2,

        max_features="sqrt",

        random_state=42,

        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    training_time = (
        time.time() - start_time
    )

    # Prediction

    prediction = model.predict(
        X_test
    )

    # Metrics

    mae = mean_absolute_error(
        y_test,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            prediction
        )
    )

    r2 = r2_score(
        y_test,
        prediction
    )

    weather_models[
        target
    ] = model

    weather_results[
        target
    ] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time_Min":
            training_time / 60
    }

    print(
        f"MAE  : {mae:.4f}"
    )

    print(
        f"RMSE : {rmse:.4f}"
    )

    print(
        f"R²   : {r2:.4f}"
    )

    print(
        f"Time : "
        f"{training_time / 60:.2f} min"
    )


# ============================================================
# 13. FINAL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    weather_results
).T.reset_index()

results_df = results_df.rename(
    columns={
        "index": "Weather_Variable"
    }
)

print("\n")
print("=" * 70)
print("WEATHER FORECASTING RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 14. PREDICT TOMORROW'S WEATHER
# ============================================================

latest_features = (
    county_df[features]
    .iloc[[-1]]
)

tomorrow_weather = {}


for target in weather_columns:

    model = weather_models[target]

    predicted_value = (
        model.predict(
            latest_features
        )[0]
    )

    tomorrow_weather[
        target
    ] = predicted_value


# ============================================================
# 15. DISPLAY FORECAST
# ============================================================

latest_date = (
    county_df["date"].iloc[-1]
)

future_date = (
    latest_date +
    pd.Timedelta(days=1)
)

print("\n")
print("=" * 70)
print("NEXT-DAY WEATHER FORECAST")
print("=" * 70)

print(
    "County FIPS :",
    TARGET_FIPS
)

print(
    "Last date   :",
    latest_date.date()
)

print(
    "Forecast for:",
    future_date.date()
)

print(
    f"Rainfall    : "
    f"{tomorrow_weather['PRECTOT']:.2f}"
)

print(
    f"Temperature : "
    f"{tomorrow_weather['T2M']:.2f} °C"
)

print(
    f"Humidity    : "
    f"{tomorrow_weather['QV2M']:.2f}"
)

print(
    "\nWeather forecast generated "
    "for reasoning only."
)

Loading dataset...


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [4]:
import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


In [5]:
# ============================================================
# 5. SEASONAL FEATURES
# ============================================================

county_df["month"] = (
    county_df["date"].dt.month
)

county_df["day_of_year"] = (
    county_df["date"].dt.dayofyear
)

# Cyclical representation of season

county_df["month_sin"] = np.sin(
    2 * np.pi *
    county_df["month"] / 12
)

county_df["month_cos"] = np.cos(
    2 * np.pi *
    county_df["month"] / 12
)

county_df["day_sin"] = np.sin(
    2 * np.pi *
    county_df["day_of_year"] / 365
)

county_df["day_cos"] = np.cos(
    2 * np.pi *
    county_df["day_of_year"] / 365
)


# ============================================================
# 6. LAG FEATURES
# ============================================================

LAGS = 30

for col in weather_columns:

    for lag in range(1, LAGS + 1):

        county_df[
            f"{col}_lag{lag}"
        ] = county_df[col].shift(lag)


# ============================================================
# 7. ROLLING WEATHER FEATURES
# ============================================================

for col in weather_columns:

    county_df[
        f"{col}_mean_7d"
    ] = (
        county_df[col]
        .rolling(7)
        .mean()
    )

    county_df[
        f"{col}_mean_14d"
    ] = (
        county_df[col]
        .rolling(14)
        .mean()
    )

    county_df[
        f"{col}_mean_30d"
    ] = (
        county_df[col]
        .rolling(30)
        .mean()
    )


# ============================================================
# 8. TARGET = NEXT DAY
# ============================================================

for col in weather_columns:

    county_df[
        f"target_{col}"
    ] = county_df[col].shift(-1)


# ============================================================
# 9. REMOVE MISSING ROWS
# ============================================================

county_df = (
    county_df
    .dropna()
    .reset_index(drop=True)
)

print(
    "\nData available for training:",
    county_df.shape
)


# ============================================================
# 10. CREATE FEATURE LIST
# ============================================================

features = []

# Lag features

for col in weather_columns:

    for lag in range(1, LAGS + 1):

        features.append(
            f"{col}_lag{lag}"
        )


# Rolling features

for col in weather_columns:

    features.extend([
        f"{col}_mean_7d",
        f"{col}_mean_14d",
        f"{col}_mean_30d"
    ])


# Seasonal features

features.extend([
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos"
])


X = county_df[features]


# ============================================================
# 11. CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================

split_index = int(
    len(county_df) * 0.80
)

X_train = X.iloc[
    :split_index
]

X_test = X.iloc[
    split_index:
]

print(
    "\nTraining rows:",
    len(X_train)
)

print(
    "Testing rows :",
    len(X_test)
)


# ============================================================
# 12. TRAIN MODELS
# ============================================================

weather_models = {}

weather_results = {}


for target in weather_columns:

    print("\n")
    print("=" * 70)
    print(
        f"TRAINING WEATHER MODEL — {target}"
    )
    print("=" * 70)

    target_column = (
        f"target_{target}"
    )

    y = county_df[target_column]

    y_train = y.iloc[
        :split_index
    ]

    y_test = y.iloc[
        split_index:
    ]

    start_time = time.time()

    model = RandomForestRegressor(

        n_estimators=150,

        max_depth=15,

        min_samples_split=2,

        min_samples_leaf=2,

        max_features="sqrt",

        random_state=42,

        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    training_time = (
        time.time() - start_time
    )

    # Prediction

    prediction = model.predict(
        X_test
    )

    # Metrics

    mae = mean_absolute_error(
        y_test,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            prediction
        )
    )

    r2 = r2_score(
        y_test,
        prediction
    )

    weather_models[
        target
    ] = model

    weather_results[
        target
    ] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time_Min":
            training_time / 60
    }

    print(
        f"MAE  : {mae:.4f}"
    )

    print(
        f"RMSE : {rmse:.4f}"
    )

    print(
        f"R²   : {r2:.4f}"
    )

    print(
        f"Time : "
        f"{training_time / 60:.2f} min"
    )


# ============================================================
# 13. FINAL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    weather_results
).T.reset_index()

results_df = results_df.rename(
    columns={
        "index": "Weather_Variable"
    }
)

print("\n")
print("=" * 70)
print("WEATHER FORECASTING RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 14. PREDICT TOMORROW'S WEATHER
# ============================================================

latest_features = (
    county_df[features]
    .iloc[[-1]]
)

tomorrow_weather = {}


for target in weather_columns:

    model = weather_models[target]

    predicted_value = (
        model.predict(
            latest_features
        )[0]
    )

    tomorrow_weather[
        target
    ] = predicted_value


# ============================================================
# 15. DISPLAY FORECAST
# ============================================================

latest_date = (
    county_df["date"].iloc[-1]
)

future_date = (
    latest_date +
    pd.Timedelta(days=1)
)

print("\n")
print("=" * 70)
print("NEXT-DAY WEATHER FORECAST")
print("=" * 70)

print(
    "County FIPS :",
    TARGET_FIPS
)

print(
    "Last date   :",
    latest_date.date()
)

print(
    "Forecast for:",
    future_date.date()
)

print(
    f"Rainfall    : "
    f"{tomorrow_weather['PRECTOT']:.2f}"
)

print(
    f"Temperature : "
    f"{tomorrow_weather['T2M']:.2f} °C"
)

print(
    f"Humidity    : "
    f"{tomorrow_weather['QV2M']:.2f}"
)

print(
    "\nWeather forecast generated "
    "for reasoning only."
)


Data available for training: (6117, 113)

Training rows: 4893
Testing rows : 1224


TRAINING WEATHER MODEL — PRECTOT
MAE  : 4.9966
RMSE : 9.2813
R²   : -0.0030
Time : 0.01 min


TRAINING WEATHER MODEL — T2M
MAE  : 2.5942
RMSE : 3.5109
R²   : 0.8387
Time : 0.01 min


TRAINING WEATHER MODEL — QV2M
MAE  : 2.0022
RMSE : 2.5302
R²   : 0.7232
Time : 0.01 min


WEATHER FORECASTING RESULTS
Weather_Variable      MAE     RMSE        R2  Training_Time_Min
         PRECTOT 4.996574 9.281280 -0.002987           0.008334
             T2M 2.594190 3.510892  0.838707           0.007927
            QV2M 2.002248 2.530230  0.723162           0.007826


NEXT-DAY WEATHER FORECAST
County FIPS : 48001
Last date   : 2016-12-28
Forecast for: 2016-12-29
Rainfall    : 4.34
Temperature : 9.87 °C
Humidity    : 6.90

Weather forecast generated for reasoning only.


In [7]:
# ============================================================
# IMPROVED WEATHER FORECASTING MODEL
# Used ONLY for drought reasoning
# ============================================================

import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_PATH = r"C:\Users\niran\Downloads\Drought prediction_ML\data\train_timeseries.csv"

print("Loading dataset...")

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)


# ============================================================
# 2. SELECT WEATHER COLUMNS
# ============================================================

required_columns = [
    "fips",
    "date",
    "PRECTOT",
    "T2M",
    "QV2M"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

df = df[required_columns].copy()


# ============================================================
# 3. CLEAN DATA
# ============================================================

df["fips"] = (
    df["fips"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

weather_columns = [
    "PRECTOT",
    "T2M",
    "QV2M"
]

for col in weather_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    df.loc[
        df[col] <= -900,
        col
    ] = np.nan


df = df.dropna(
    subset=weather_columns
)

df = df.sort_values(
    ["fips", "date"]
).reset_index(drop=True)

print("After cleaning:", df.shape)


# ============================================================
# 4. SELECT ONE COUNTY FOR TESTING
# ============================================================

TARGET_FIPS = "48001"

county_df = df[
    df["fips"] == TARGET_FIPS
].copy()

county_df = county_df.sort_values(
    "date"
).reset_index(drop=True)

print("\nCounty:", TARGET_FIPS)
print("County shape:", county_df.shape)


# ============================================================
# 5. SEASONAL FEATURES
# ============================================================

county_df["month"] = (
    county_df["date"].dt.month
)

county_df["day_of_year"] = (
    county_df["date"].dt.dayofyear
)

# Cyclical representation of season

county_df["month_sin"] = np.sin(
    2 * np.pi *
    county_df["month"] / 12
)

county_df["month_cos"] = np.cos(
    2 * np.pi *
    county_df["month"] / 12
)

county_df["day_sin"] = np.sin(
    2 * np.pi *
    county_df["day_of_year"] / 365
)

county_df["day_cos"] = np.cos(
    2 * np.pi *
    county_df["day_of_year"] / 365
)


# ============================================================
# 6. LAG FEATURES
# ============================================================

LAGS = 30

for col in weather_columns:

    for lag in range(1, LAGS + 1):

        county_df[
            f"{col}_lag{lag}"
        ] = county_df[col].shift(lag)


# ============================================================
# 7. ROLLING WEATHER FEATURES
# ============================================================

for col in weather_columns:

    county_df[
        f"{col}_mean_7d"
    ] = (
        county_df[col]
        .rolling(7)
        .mean()
    )

    county_df[
        f"{col}_mean_14d"
    ] = (
        county_df[col]
        .rolling(14)
        .mean()
    )

    county_df[
        f"{col}_mean_30d"
    ] = (
        county_df[col]
        .rolling(30)
        .mean()
    )


# ============================================================
# 8. TARGET = NEXT DAY
# ============================================================

for col in weather_columns:

    county_df[
        f"target_{col}"
    ] = county_df[col].shift(-1)


# ============================================================
# 9. REMOVE MISSING ROWS
# ============================================================

county_df = (
    county_df
    .dropna()
    .reset_index(drop=True)
)

print(
    "\nData available for training:",
    county_df.shape
)


# ============================================================
# 10. CREATE FEATURE LIST
# ============================================================

features = []

# Lag features

for col in weather_columns:

    for lag in range(1, LAGS + 1):

        features.append(
            f"{col}_lag{lag}"
        )


# Rolling features

for col in weather_columns:

    features.extend([
        f"{col}_mean_7d",
        f"{col}_mean_14d",
        f"{col}_mean_30d"
    ])


# Seasonal features

features.extend([
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos"
])


X = county_df[features]


# ============================================================
# 11. CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================

split_index = int(
    len(county_df) * 0.80
)

X_train = X.iloc[
    :split_index
]

X_test = X.iloc[
    split_index:
]

print(
    "\nTraining rows:",
    len(X_train)
)

print(
    "Testing rows :",
    len(X_test)
)


# ============================================================
# 12. TRAIN MODELS
# ============================================================

weather_models = {}

weather_results = {}


for target in weather_columns:

    print("\n")
    print("=" * 70)
    print(
        f"TRAINING WEATHER MODEL — {target}"
    )
    print("=" * 70)

    target_column = (
        f"target_{target}"
    )

    y = county_df[target_column]

    y_train = y.iloc[
        :split_index
    ]

    y_test = y.iloc[
        split_index:
    ]

    start_time = time.time()

    model = RandomForestRegressor(

        n_estimators=150,

        max_depth=15,

        min_samples_split=2,

        min_samples_leaf=2,

        max_features="sqrt",

        random_state=42,

        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    training_time = (
        time.time() - start_time
    )

    # Prediction

    prediction = model.predict(
        X_test
    )

    # Metrics

    mae = mean_absolute_error(
        y_test,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            prediction
        )
    )

    r2 = r2_score(
        y_test,
        prediction
    )

    weather_models[
        target
    ] = model

    weather_results[
        target
    ] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time_Min":
            training_time / 60
    }

    print(
        f"MAE  : {mae:.4f}"
    )

    print(
        f"RMSE : {rmse:.4f}"
    )

    print(
        f"R²   : {r2:.4f}"
    )

    print(
        f"Time : "
        f"{training_time / 60:.2f} min"
    )


# ============================================================
# 13. FINAL RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    weather_results
).T.reset_index()

results_df = results_df.rename(
    columns={
        "index": "Weather_Variable"
    }
)

print("\n")
print("=" * 70)
print("WEATHER FORECASTING RESULTS")
print("=" * 70)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 14. PREDICT TOMORROW'S WEATHER
# ============================================================

latest_features = (
    county_df[features]
    .iloc[[-1]]
)

tomorrow_weather = {}


for target in weather_columns:

    model = weather_models[target]

    predicted_value = (
        model.predict(
            latest_features
        )[0]
    )

    tomorrow_weather[
        target
    ] = predicted_value


# ============================================================
# 15. DISPLAY FORECAST
# ============================================================

latest_date = (
    county_df["date"].iloc[-1]
)

future_date = (
    latest_date +
    pd.Timedelta(days=1)
)

print("\n")
print("=" * 70)
print("NEXT-DAY WEATHER FORECAST")
print("=" * 70)

print(
    "County FIPS :",
    TARGET_FIPS
)

print(
    "Last date   :",
    latest_date.date()
)

print(
    "Forecast for:",
    future_date.date()
)

print(
    f"Rainfall    : "
    f"{tomorrow_weather['PRECTOT']:.2f}"
)

print(
    f"Temperature : "
    f"{tomorrow_weather['T2M']:.2f} °C"
)

print(
    f"Humidity    : "
    f"{tomorrow_weather['QV2M']:.2f}"
)

print(
    "\nWeather forecast generated "
    "for reasoning only."
)

Loading dataset...
Original shape: (19300680, 21)
After cleaning: (19300680, 5)

County: 48001
County shape: (6210, 5)


C:\Users\niran\AppData\Local\Temp\ipykernel_35960\2924347936.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  county_df[
C:\Users\niran\AppData\Local\Temp\ipykernel_35960\2924347936.py:176: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  county_df[
C:\Users\niran\AppData\Local\Temp\ipykernel_35960\2924347936.py:184: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) inste


Data available for training: (6179, 113)

Training rows: 4943
Testing rows : 1236


TRAINING WEATHER MODEL — PRECTOT
MAE  : 4.9599
RMSE : 9.2560
R²   : -0.0058
Time : 0.01 min


TRAINING WEATHER MODEL — T2M
MAE  : 2.5975
RMSE : 3.4976
R²   : 0.8402
Time : 0.01 min


TRAINING WEATHER MODEL — QV2M
MAE  : 2.0097
RMSE : 2.5338
R²   : 0.7211
Time : 0.01 min


WEATHER FORECASTING RESULTS
Weather_Variable      MAE     RMSE        R2  Training_Time_Min
         PRECTOT 4.959871 9.255963 -0.005846           0.008864
             T2M 2.597450 3.497648  0.840198           0.007783
            QV2M 2.009682 2.533802  0.721139           0.007010


NEXT-DAY WEATHER FORECAST
County FIPS : 48001
Last date   : 2016-12-30
Forecast for: 2016-12-31
Rainfall    : 4.67
Temperature : 12.12 °C
Humidity    : 7.50

Weather forecast generated for reasoning only.


In [2]:
# ============================================================
# IMPROVED WEATHER FORECASTING SYSTEM
# WEATHER IS USED ONLY FOR DROUGHT REASONING
# ============================================================

import pandas as pd
import numpy as np
import time

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
    classification_report
)


# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_PATH = r"C:\Users\niran\Downloads\Drought prediction_ML\data\train_timeseries.csv"

print("=" * 70)
print("LOADING WEATHER DATA")
print("=" * 70)

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)


# ============================================================
# 2. WEATHER FEATURES
# ============================================================

weather_columns = [
    "PRECTOT",
    "PS",
    "QV2M",
    "T2M",
    "T2MDEW",
    "T2MWET",
    "T2M_MAX",
    "T2M_MIN",
    "T2M_RANGE",
    "TS",
    "WS10M",
    "WS10M_MAX",
    "WS10M_MIN",
    "WS10M_RANGE",
    "WS50M",
    "WS50M_MAX",
    "WS50M_MIN",
    "WS50M_RANGE"
]

required_columns = [
    "fips",
    "date"
] + weather_columns


missing = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing:

    raise ValueError(
        f"Missing columns: {missing}"
    )


df = df[required_columns].copy()


# ============================================================
# 3. CLEAN DATA
# ============================================================

df["fips"] = (
    df["fips"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

for col in weather_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    # NASA missing values
    df.loc[
        df[col] <= -900,
        col
    ] = np.nan


df = df.dropna(
    subset=weather_columns
)

df = df.sort_values(
    ["fips", "date"]
).reset_index(drop=True)

print(
    "After cleaning:",
    df.shape
)


# ============================================================
# 4. SELECT ONE COUNTY
# ============================================================

TARGET_FIPS = "48001"

county = df[
    df["fips"] == TARGET_FIPS
].copy()

county = county.sort_values(
    "date"
).reset_index(drop=True)

print()
print("=" * 70)
print("COUNTY DATA")
print("=" * 70)

print(
    "FIPS:",
    TARGET_FIPS
)

print(
    "Shape:",
    county.shape
)

print(
    "Date:",
    county["date"].min(),
    "->",
    county["date"].max()
)


# ============================================================
# 5. SEASONAL FEATURES
# ============================================================

county["month"] = (
    county["date"].dt.month
)

county["day_of_year"] = (
    county["date"].dt.dayofyear
)

county["year"] = (
    county["date"].dt.year
)

county["month_sin"] = np.sin(
    2 * np.pi *
    county["month"] / 12
)

county["month_cos"] = np.cos(
    2 * np.pi *
    county["month"] / 12
)

county["day_sin"] = np.sin(
    2 * np.pi *
    county["day_of_year"] / 365
)

county["day_cos"] = np.cos(
    2 * np.pi *
    county["day_of_year"] / 365
)


# ============================================================
# 6. LAG FEATURES
# ============================================================

print()
print("Creating lag features...")

lags = [
    1,
    3,
    7,
    14,
    30,
    60
]

# We don't need every variable at every lag.
# Use the most important weather variables.

lag_columns = [
    "PRECTOT",
    "T2M",
    "QV2M",
    "T2M_MAX",
    "T2M_MIN",
    "WS10M"
]

for col in lag_columns:

    for lag in lags:

        county[
            f"{col}_lag{lag}"
        ] = county[col].shift(lag)


# ============================================================
# 7. RAINFALL-SPECIFIC FEATURES
# ============================================================

print("Creating rainfall features...")

county["rain_3d"] = (
    county["PRECTOT"]
    .rolling(3)
    .sum()
)

county["rain_7d"] = (
    county["PRECTOT"]
    .rolling(7)
    .sum()
)

county["rain_14d"] = (
    county["PRECTOT"]
    .rolling(14)
    .sum()
)

county["rain_30d"] = (
    county["PRECTOT"]
    .rolling(30)
    .sum()
)

county["rain_mean_7d"] = (
    county["PRECTOT"]
    .rolling(7)
    .mean()
)

county["rain_mean_30d"] = (
    county["PRECTOT"]
    .rolling(30)
    .mean()
)

county["rain_max_7d"] = (
    county["PRECTOT"]
    .rolling(7)
    .max()
)

county["rain_max_30d"] = (
    county["PRECTOT"]
    .rolling(30)
    .max()
)


# ============================================================
# 8. DAYS SINCE RAIN
# ============================================================

rain_threshold = 1.0

rain_event = (
    county["PRECTOT"] >= rain_threshold
)

days_since_rain = []

counter = 0

for value in rain_event:

    if value:

        counter = 0

    else:

        counter += 1

    days_since_rain.append(counter)


county["days_since_rain"] = (
    days_since_rain
)


# ============================================================
# 9. TARGETS
# ============================================================

# Tomorrow's values

county["target_T2M"] = (
    county["T2M"].shift(-1)
)

county["target_QV2M"] = (
    county["QV2M"].shift(-1)
)

county["target_PRECTOT"] = (
    county["PRECTOT"].shift(-1)
)


# Rain occurrence target

county["target_rain_event"] = (
    county["target_PRECTOT"] >= rain_threshold
).astype(int)


# ============================================================
# 10. REMOVE NaN
# ============================================================

county = (
    county
    .dropna()
    .reset_index(drop=True)
)

print()
print(
    "Data available:",
    county.shape
)


# ============================================================
# 11. CREATE FEATURE LIST
# ============================================================

features = []

# Lag features

for col in lag_columns:

    for lag in lags:

        features.append(
            f"{col}_lag{lag}"
        )


# Rainfall features

features.extend([
    "rain_3d",
    "rain_7d",
    "rain_14d",
    "rain_30d",
    "rain_mean_7d",
    "rain_mean_30d",
    "rain_max_7d",
    "rain_max_30d",
    "days_since_rain"
])


# Seasonal features

features.extend([
    "month",
    "day_of_year",
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos"
])


# Current atmospheric conditions

features.extend([
    "PS",
    "T2M",
    "T2MDEW",
    "T2MWET",
    "T2M_MAX",
    "T2M_MIN",
    "T2M_RANGE",
    "TS",
    "WS10M",
    "WS10M_MAX",
    "WS10M_MIN",
    "WS10M_RANGE",
    "WS50M",
    "WS50M_MAX",
    "WS50M_MIN",
    "WS50M_RANGE"
])


X = county[features]


# ============================================================
# 12. CHRONOLOGICAL SPLIT
# ============================================================

split = int(
    len(county) * 0.80
)

X_train = X.iloc[
    :split
]

X_test = X.iloc[
    split:
]

print()
print(
    "Training rows:",
    len(X_train)
)

print(
    "Testing rows:",
    len(X_test)
)


# ============================================================
# 13. TEMPERATURE MODEL
# ============================================================

print()
print("=" * 70)
print("TEMPERATURE MODEL")
print("=" * 70)

y_train = county[
    "target_T2M"
].iloc[:split]

y_test = county[
    "target_T2M"
].iloc[split:]

start = time.time()

temperature_model = RandomForestRegressor(

    n_estimators=200,

    max_depth=20,

    min_samples_leaf=2,

    max_features="sqrt",

    random_state=42,

    n_jobs=-1
)

temperature_model.fit(
    X_train,
    y_train
)

temperature_pred = (
    temperature_model.predict(
        X_test
    )
)

temperature_mae = (
    mean_absolute_error(
        y_test,
        temperature_pred
    )
)

temperature_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        temperature_pred
    )
)

temperature_r2 = r2_score(
    y_test,
    temperature_pred
)

print(
    f"MAE  : {temperature_mae:.4f} °C"
)

print(
    f"RMSE : {temperature_rmse:.4f} °C"
)

print(
    f"R²   : {temperature_r2:.4f}"
)

print(
    f"Time : {(time.time()-start)/60:.2f} min"
)


# ============================================================
# 14. HUMIDITY MODEL
# ============================================================

print()
print("=" * 70)
print("HUMIDITY MODEL")
print("=" * 70)

y_train = county[
    "target_QV2M"
].iloc[:split]

y_test = county[
    "target_QV2M"
].iloc[split:]

start = time.time()

humidity_model = RandomForestRegressor(

    n_estimators=200,

    max_depth=20,

    min_samples_leaf=2,

    max_features="sqrt",

    random_state=42,

    n_jobs=-1
)

humidity_model.fit(
    X_train,
    y_train
)

humidity_pred = (
    humidity_model.predict(
        X_test
    )
)

humidity_mae = (
    mean_absolute_error(
        y_test,
        humidity_pred
    )
)

humidity_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        humidity_pred
    )
)

humidity_r2 = r2_score(
    y_test,
    humidity_pred
)

print(
    f"MAE  : {humidity_mae:.4f}"
)

print(
    f"RMSE : {humidity_rmse:.4f}"
)

print(
    f"R²   : {humidity_r2:.4f}"
)

print(
    f"Time : {(time.time()-start)/60:.2f} min"
)


# ============================================================
# 15. RAIN OCCURRENCE MODEL
# ============================================================

print()
print("=" * 70)
print("RAINFALL OCCURRENCE MODEL")
print("=" * 70)

y_train = county[
    "target_rain_event"
].iloc[:split]

y_test = county[
    "target_rain_event"
].iloc[split:]

start = time.time()

rain_classifier = RandomForestClassifier(

    n_estimators=200,

    max_depth=20,

    min_samples_leaf=2,

    class_weight="balanced",

    max_features="sqrt",

    random_state=42,

    n_jobs=-1
)

rain_classifier.fit(
    X_train,
    y_train
)

rain_event_pred = (
    rain_classifier.predict(
        X_test
    )
)

rain_accuracy = (
    accuracy_score(
        y_test,
        rain_event_pred
    )
)

rain_f1 = (
    f1_score(
        y_test,
        rain_event_pred,
        average="weighted",
        zero_division=0
    )
)

print(
    f"Accuracy : {rain_accuracy:.4f}"
)

print(
    f"Weighted F1 : {rain_f1:.4f}"
)

print(
    f"Time : {(time.time()-start)/60:.2f} min"
)

print()
print(
    classification_report(
        y_test,
        rain_event_pred,
        zero_division=0
    )
)


# ============================================================
# 16. RAINFALL AMOUNT MODEL
# ============================================================

print()
print("=" * 70)
print("RAINFALL AMOUNT MODEL")
print("=" * 70)

# Only train amount model on days where
# rainfall actually occurred.

rain_train_mask = (
    county[
        "target_rain_event"
    ].iloc[:split] == 1
)

X_rain_train = X_train[
    rain_train_mask.values
]

y_rain_train = county[
    "target_PRECTOT"
].iloc[:split][
    rain_train_mask.values
]

# Test only on actual rain events

rain_test_mask = (
    county[
        "target_rain_event"
    ].iloc[split:] == 1
)

X_rain_test = X_test[
    rain_test_mask.values
]

y_rain_test = county[
    "target_PRECTOT"
].iloc[split:][
    rain_test_mask.values
]

print(
    "Rainfall training rows:",
    len(X_rain_train)
)

print(
    "Rainfall test rows:",
    len(X_rain_test)
)


rain_amount_model = RandomForestRegressor(

    n_estimators=200,

    max_depth=20,

    min_samples_leaf=2,

    max_features="sqrt",

    random_state=42,

    n_jobs=-1
)

start = time.time()

rain_amount_model.fit(
    X_rain_train,
    y_rain_train
)

if len(X_rain_test) > 0:

    rain_amount_pred = (
        rain_amount_model.predict(
            X_rain_test
        )
    )

    rain_amount_mae = (
        mean_absolute_error(
            y_rain_test,
            rain_amount_pred
        )
    )

    rain_amount_rmse = np.sqrt(
        mean_squared_error(
            y_rain_test,
            rain_amount_pred
        )
    )

    rain_amount_r2 = r2_score(
        y_rain_test,
        rain_amount_pred
    )

    print(
        f"MAE  : {rain_amount_mae:.4f}"
    )

    print(
        f"RMSE : {rain_amount_rmse:.4f}"
    )

    print(
        f"R²   : {rain_amount_r2:.4f}"
    )

else:

    print(
        "No rainy samples in test set."
    )

print(
    f"Time : {(time.time()-start)/60:.2f} min"
)


# ============================================================
# 17. PREDICT TOMORROW
# ============================================================

latest_X = (
    county[features]
    .iloc[[-1]]
)


predicted_temperature = (
    temperature_model
    .predict(latest_X)[0]
)

predicted_humidity = (
    humidity_model
    .predict(latest_X)[0]
)

rain_event = (
    rain_classifier
    .predict(latest_X)[0]
)


if rain_event == 1:

    predicted_rainfall = (
        rain_amount_model
        .predict(latest_X)[0]
    )

    predicted_rainfall = max(
        0,
        predicted_rainfall
    )

else:

    predicted_rainfall = 0.0


# ============================================================
# 18. WEATHER CONDITION CATEGORIES
# ============================================================

# Rainfall category

if predicted_rainfall < 1:

    rainfall_condition = "LOW"

elif predicted_rainfall < 10:

    rainfall_condition = "NORMAL"

else:

    rainfall_condition = "HIGH"


# Temperature category
# These are simple reasoning thresholds.
# We can later make them county/season specific.

if predicted_temperature >= 30:

    temperature_condition = "HIGH"

elif predicted_temperature <= 15:

    temperature_condition = "LOW"

else:

    temperature_condition = "NORMAL"


# Humidity / QV2M category

if predicted_humidity < 5:

    humidity_condition = "LOW"

elif predicted_humidity > 12:

    humidity_condition = "HIGH"

else:

    humidity_condition = "NORMAL"


# ============================================================
# 19. FINAL FORECAST
# ============================================================

latest_date = (
    county["date"].iloc[-1]
)

forecast_date = (
    latest_date +
    pd.Timedelta(days=1)
)


print()
print("=" * 70)
print("FINAL NEXT-DAY WEATHER FORECAST")
print("=" * 70)

print(
    "County FIPS :",
    TARGET_FIPS
)

print(
    "Forecast date:",
    forecast_date.date()
)

print()

print(
    f"Rainfall    : "
    f"{predicted_rainfall:.2f}"
)

print(
    f"Rainfall condition : "
    f"{rainfall_condition}"
)

print()

print(
    f"Temperature : "
    f"{predicted_temperature:.2f} °C"
)

print(
    f"Temperature condition : "
    f"{temperature_condition}"
)

print()

print(
    f"QV2M        : "
    f"{predicted_humidity:.2f}"
)

print(
    f"Humidity condition : "
    f"{humidity_condition}"
)


# ============================================================
# 20. REASONING
# ============================================================

reasons = []

if rainfall_condition == "LOW":

    reasons.append(
        "low expected rainfall"
    )

if temperature_condition == "HIGH":

    reasons.append(
        "high expected temperature"
    )

if humidity_condition == "LOW":

    reasons.append(
        "low expected humidity"
    )


if len(reasons) == 0:

    reasoning = (
        "Predicted weather conditions "
        "do not show strong additional "
        "drought stress signals."
    )

else:

    reasoning = (
        "The forecast indicates "
        + ", ".join(reasons)
        + ", which may increase "
          "drought stress."
    )


print()
print("=" * 70)
print("DROUGHT REASONING")
print("=" * 70)

print(reasoning)

LOADING WEATHER DATA
Original shape: (19300680, 21)
After cleaning: (19300680, 20)

COUNTY DATA
FIPS: 48001
Shape: (6210, 20)
Date: 2000-01-01 00:00:00 -> 2016-12-31 00:00:00

Creating lag features...
Creating rainfall features...

Data available: (6149, 76)

Training rows: 4919
Testing rows: 1230

TEMPERATURE MODEL
MAE  : 1.6426 °C
RMSE : 2.3613 °C
R²   : 0.9272
Time : 0.02 min

HUMIDITY MODEL
MAE  : 1.3472
RMSE : 1.7965
R²   : 0.8603
Time : 0.01 min

RAINFALL OCCURRENCE MODEL
Accuracy : 0.7593
Weighted F1 : 0.7579
Time : 0.02 min

              precision    recall  f1-score   support

           0       0.82      0.83      0.83       848
           1       0.62      0.59      0.60       382

    accuracy                           0.76      1230
   macro avg       0.72      0.71      0.72      1230
weighted avg       0.76      0.76      0.76      1230


RAINFALL AMOUNT MODEL
Rainfall training rows: 1580
Rainfall test rows: 382
MAE  : 8.6170
RMSE : 13.3508
R²   : 0.0632
Time : 0.01 min